##Step 1: Clean the parking violations dataset

**Input:   jan_to_may_police_violation_anonymized791b166.csv              
Output:  clean_parking.csv**

In [ ]:
import pandas as pd
RAW_FILE = "jan to may police violation_anonymized791b166.csv"
OUT_FILE = "clean_parking.csv"

In [ ]:
df=pd.read_csv(RAW_FILE)

In [ ]:
df.head()

,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,...,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,...,82.0,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,...,26.0,Byatarayanapura,True,No Junction,NaN,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,...,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,NaN,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00


In [ ]:
df.shape

(298450, 24)

In [ ]:
df.describe()

,latitude,longitude,description,closed_datetime,center_code,action_taken_timestamp
count,298450.000000,298450.000000,0.0,0.0,287190.000000,0.0
mean,12.980802,77.600512,NaN,NaN,23.023013,NaN
std,0.049732,0.050518,NaN,NaN,20.006118,NaN
min,12.802667,77.442553,NaN,NaN,2.000000,NaN
25%,12.963331,77.571198,NaN,NaN,11.000000,NaN
50%,12.977284,77.584114,NaN,NaN,17.000000,NaN
75%,12.997467,77.621529,NaN,NaN,29.000000,NaN
max,13.293684,77.771735,NaN,NaN,88.000000,NaN


In [ ]:
# Number of missing values in each column
df.isnull().sum()

,0
id,0
latitude,0
longitude,0
location,3041
vehicle_number,0
vehicle_type,0
description,298450
violation_type,0
offence_code,0
created_datetime,0


In [ ]:
# 1. Drop the 3 fully-empty columns

EMPTY_COLS = ["description", "closed_datetime", "action_taken_timestamp"]
df = df.drop(columns=EMPTY_COLS)
print(f"Dropped {EMPTY_COLS}. Shape now: {df.shape}")

Dropped ['description', 'closed_datetime', 'action_taken_timestamp']. Shape now: (298450, 21)


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["validation_status"].value_counts(dropna=False)

,count
validation_status,
NaN,125254
approved,115400
rejected,49754
created1,7044
processing,678
duplicate,320


In [ ]:
print("\nvalidation_status distribution before filtering:")
print(df["validation_status"].value_counts(dropna=False))

before = len(df)
df = df[~df["validation_status"].isin(["rejected", "duplicate"])]
print(f"\nDropped rejected+duplicate rows: {before} -> {len(df)} "
      f"(removed {before - len(df)})")


validation_status distribution before filtering:
validation_status
NaN           125254
approved      115400
rejected       49754
created1        7044
processing       678
duplicate        320
Name: count, dtype: int64

Dropped rejected+duplicate rows: 298450 -> 248376 (removed 50074)


In [ ]:
# 3. Convert created_datetime from UTC to IST (+5:30)

df["created_datetime"] = pd.to_datetime(df["created_datetime"], utc=True, format="ISO8601")
df["created_datetime_ist"] = df["created_datetime"] + pd.Timedelta(hours=5, minutes=30)

In [ ]:
# also convert modified_datetime and validation_timestamp for consistency downstream
for col in ["modified_datetime", "validation_timestamp"]:
    if col in df.columns:
        df[col + "_ist"] = pd.to_datetime(df[col], utc=True, errors="coerce", format="ISO8601") + pd.Timedelta(hours=5, minutes=30)

In [ ]:
# convenience columns used later for time-of-day analysis (CRS / dashboard charts)
df["hour_ist"] = df["created_datetime_ist"].dt.hour
df["day_of_week"] = df["created_datetime_ist"].dt.day_name()
df["date_ist"] = df["created_datetime_ist"].dt.date

print(f"\nFinal clean_parking shape: {df.shape}")
print(f"Expected ~248K rows per project plan -> got {len(df):,}")

df.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")



Final clean_parking shape: (248376, 27)
Expected ~248K rows per project plan -> got 248,376

Saved -> clean_parking.csv


## Step 2: Clean the Astram incidents dataset

**Input:  Astram_event_data_anonymized_-_Astram_event_data_anonymizedb40ac87.csv  
Output: clean_incidents.csv**

In [ ]:
import pandas as pd

RAW_FILE = "Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"
OUT_FILE = "clean_incidents.csv"

In [ ]:
data = pd.read_csv(RAW_FILE)
data.head()

,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


In [ ]:
data.shape

(8173, 46)

In [ ]:
data.describe()

,latitude,longitude,endlatitude,endlongitude,map_file,age_of_truck,client_id,comment,meta_data,resolved_at_latitude,resolved_at_longitude
count,8173.000000,8173.000000,8004.000000,8004.000000,0.0,276.000000,8173.000000,0.0,0.0,74.000000,74.000000
mean,12.987076,77.596034,1.128050,6.678011,NaN,235.518116,1.009788,NaN,NaN,13.002599,77.569049
std,0.060109,0.061193,3.736845,21.761316,NaN,634.059816,0.098457,NaN,NaN,0.091377,0.057246
min,12.801041,77.308731,0.000000,0.000000,NaN,0.000000,1.000000,NaN,NaN,12.841568,77.390820
25%,12.951635,77.556747,0.000000,0.000000,NaN,5.000000,1.000000,NaN,NaN,12.947656,77.543782
50%,12.982847,77.589460,0.000000,0.000000,NaN,10.000000,1.000000,NaN,NaN,12.983715,77.556135
75%,13.026867,77.625853,0.000000,0.000000,NaN,15.000000,1.000000,NaN,NaN,13.019590,77.589899
max,13.267510,77.769403,59.860133,80.720691,NaN,2026.000000,2.000000,NaN,NaN,13.257297,77.712662


In [ ]:
data.isnull().sum()

,0
id,0
event_type,0
latitude,0
longitude,0
endlatitude,169
endlongitude,169
address,3
end_address,7486
event_cause,0
requires_road_closure,0


In [ ]:
# 1. Drop the 3 fully-empty columns

EMPTY_COLS = ["map_file", "comment", "meta_data"]
data = data.drop(columns=EMPTY_COLS)
print(f"Dropped {EMPTY_COLS}. Shape now: {data.shape}")

Dropped ['map_file', 'comment', 'meta_data']. Shape now: (8173, 43)


In [ ]:
data["authenticated"].value_counts()

,count
authenticated,
yes,7166
no,1007


In [ ]:
# 2. Filter to authenticated == 'yes' only

print("\nauthenticated distribution before filtering:")
print(data["authenticated"].value_counts(dropna=False))

before = len(data)
data = data[data["authenticated"] == "yes"]
print(f"\nFiltered to authenticated=='yes': {before} -> {len(data)}")


authenticated distribution before filtering:
authenticated
yes    7166
no     1007
Name: count, dtype: int64

Filtered to authenticated=='yes': 8173 -> 7166


In [ ]:
data.shape

(7166, 43)

In [ ]:
# 3. Compute incident_duration = closed_datetime - start_datetime

data["start_datetime"] = pd.to_datetime(data["start_datetime"], utc=True, format="ISO8601")
data["closed_datetime"] = pd.to_datetime(data["closed_datetime"], utc=True, format="ISO8601", errors="coerce")

data["incident_duration_min"] = (data["closed_datetime"] - data["start_datetime"]).dt.total_seconds() / 60

print(f"\nRows with computable duration: {data['incident_duration_min'].notna().sum()}")
print(data["incident_duration_min"].describe())


Rows with computable duration: 2412
count      2412.000000
mean       7997.667869
std       22568.349071
min      -40318.884001
25%          37.317887
50%          82.508112
75%        2077.343699
max      201789.492498
Name: incident_duration_min, dtype: float64


In [ ]:
# Drop durations over 72 hours (4320 min) AND negative/zero durations as data-entry errors (closed_datetime before start_datetime is impossible).
# Only applies to rows where duration is known; NaN rows are left alone (as they simply have no duration yet, not an invalid one).

before = len(data)
bad_duration_mask = (data["incident_duration_min"] > 72 * 60) | (data["incident_duration_min"] <= 0)
print(f"\nRows with duration > 72hr or <= 0 (data entry errors): {bad_duration_mask.sum()}")
data = data[~bad_duration_mask]
print(f"Dropped bad-duration rows: {before} -> {len(data)}")


Rows with duration > 72hr or <= 0 (data entry errors): 527
Dropped bad-duration rows: 7166 -> 6639


In [ ]:
# 4. Convert key datetimes to IST

data["start_datetime_ist"] = data["start_datetime"] + pd.Timedelta(hours=5, minutes=30)
data["closed_datetime_ist"] = data["closed_datetime"] + pd.Timedelta(hours=5, minutes=30)

for col in ["modified_datetime", "created_date", "end_datetime", "resolved_datetime"]:
    if col in data.columns:
        data[col + "_ist"] = pd.to_datetime(data[col], utc=True, errors="coerce", format="ISO8601") + pd.Timedelta(hours=5, minutes=30)

# convenience columns for later time-of-day analysis
data["hour_ist"] = data["start_datetime_ist"].dt.hour
data["day_of_week"] = data["start_datetime_ist"].dt.day_name()
data["date_ist"] = data["start_datetime_ist"].dt.date

print(f"\nFinal clean_incidents shape: {data.shape}")
print(f"Expected ~7,800-8,000 rows per project plan -> got {len(data):,}")

data.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")


Final clean_incidents shape: (6639, 53)
Expected ~7,800-8,000 rows per project plan -> got 6,639

Saved -> clean_incidents.csv


# Step 3: Parse violation_type and offence_code JSON fields
**Input:  clean_parking.csv                    
Output: parking_features.csv**

Both fields are stored as JSON arrays (e.g. "[112,107]" and
'["WRONG PARKING","PARKING IN A MAIN ROAD"]').                                 We unnest offence_code into binary severity flags used by the CRS formula:   
  

*   107 = main_road_parking
*   113 = no_parking

*   112 = wrong_parking
*   105 = footpath_parking




  
  
  

In [ ]:
import json
import pandas as pd

IN_FILE = "clean_parking.csv"
OUT_FILE = "parking_features.csv"

print("Loading clean_parking.csv...")
df = pd.read_csv(IN_FILE)
print(f"Shape: {df.shape}")

Loading clean_parking.csv...
Shape: (248376, 27)


In [ ]:
def safe_parse(x):
    """Parse a JSON-array string into a python list. Returns [] on failure/NaN."""
    if pd.isna(x):
        return []
    try:
        return json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return []


df["offence_code_parsed"] = df["offence_code"].apply(safe_parse)
df["violation_type_parsed"] = df["violation_type"].apply(safe_parse)

In [ ]:
# sanity check parse success rate
parse_fail = (df["offence_code"].notna()) & (df["offence_code_parsed"].apply(len) == 0)
print(f"offence_code parse failures: {parse_fail.sum()} / {len(df)}")


offence_code parse failures: 0 / 248376


In [ ]:
# One-hot encode the 4 severity-relevant offence codes
# ---------------------------------------------------------------
CODE_MAP = {
    107: "is_main_road_parking",
    113: "is_no_parking",
    112: "is_wrong_parking",
    105: "is_footpath_parking",
}

for code, colname in CODE_MAP.items():
    df[colname] = df["offence_code_parsed"].apply(lambda lst: 1 if code in lst else 0)

print("\nFlag counts (rows can have multiple flags = 1):")
for colname in CODE_MAP.values():
    print(f"  {colname}: {df[colname].sum():,}")


Flag counts (rows can have multiple flags = 1):
  is_main_road_parking: 19,980
  is_no_parking: 118,442
  is_wrong_parking: 134,372
  is_footpath_parking: 3,108


In [ ]:
# also keep a count of total offence codes per violation (useful as a severity multiplier signal later)
df["num_offence_codes"] = df["offence_code_parsed"].apply(len)


In [ ]:
# drop helper columns we don't need to persist (keep parsed list off disk, CSV would just re-stringify it anyway -- keep the original raw columns for traceability instead)
df = df.drop(columns=["offence_code_parsed", "violation_type_parsed"])

print(f"\nFinal parking_features shape: {df.shape}")
print(f"New columns added: {list(CODE_MAP.values()) + ['num_offence_codes']}")

df.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")


Final parking_features shape: (248376, 32)
New columns added: ['is_main_road_parking', 'is_no_parking', 'is_wrong_parking', 'is_footpath_parking', 'num_offence_codes']

Saved -> parking_features.csv


## Step 4: Build vehicle recidivism scores

**Input:  parking_feature.csv                                            Output: recidivism_scores.csv**

Groups by vehicle_number, counts violations per vehicle, and assigns a
recidivism multiplier tier used later in the CRS formula:         
  1 violation     -> 1.0x        
  2-5 violations   -> 1.5x        
  6-20 violations  -> 2.0x        
  20+ violations    -> 3.0x

In [ ]:
import pandas as pd

IN_FILE = "parking_features.csv"
OUT_FILE = "recidivism_scores.csv"

print("Loading parking_features.csv...")
df = pd.read_csv(IN_FILE)
print(f"Shape: {df.shape}")

Loading parking_features.csv...
Shape: (248376, 32)


In [ ]:
# Count violations per vehicle

veh_counts = (
    df.groupby("vehicle_number")
    .size()
    .reset_index(name="violation_count")
)

repeat_offenders = (veh_counts["violation_count"] > 1).sum()
print(f"\nTotal unique vehicles: {len(veh_counts):,}")
print(f"Vehicles with more than 1 violation: {repeat_offenders:,}")
print(f"Max violations by a single vehicle: {veh_counts['violation_count'].max()}")


Total unique vehicles: 197,260
Vehicles with more than 1 violation: 27,971
Max violations by a single vehicle: 47


In [ ]:
def recidivism_tier(count: int) -> float:
    if count == 1:
        return 1.0
    elif count <= 5:
        return 1.5
    elif count <= 20:
        return 2.0
    else:
        return 3.0


veh_counts["recidivism_tier"] = veh_counts["violation_count"].apply(recidivism_tier)

print("\nRecidivism tier distribution (by vehicle):")
print(veh_counts["recidivism_tier"].value_counts().sort_index())


Recidivism tier distribution (by vehicle):
recidivism_tier
1.0    169289
1.5     26205
2.0      1733
3.0        33
Name: count, dtype: int64


In [ ]:
# Merge back onto the full violation-level dataset
# ---------------------------------------------------------------
final = df.merge(veh_counts, on="vehicle_number", how="left")

print(f"\nFinal recidivism_scores shape: {final.shape}")
assert final["recidivism_tier"].isna().sum() == 0, "merge produced unmatched rows!"

final.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")


Final recidivism_scores shape: (248376, 34)

Saved -> recidivism_scores.csv


## Exploratory Data Analysis on cleaned datasets.

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("eda_outputs", exist_ok=True)

parking = pd.read_csv("recidivism_scores.csv")
incidents = pd.read_csv("clean_incidents.csv")

parking["created_datetime_ist"] = pd.to_datetime(parking["created_datetime_ist"], format="ISO8601")
incidents["start_datetime_ist"] = pd.to_datetime(incidents["start_datetime_ist"], format="ISO8601")

print("=" * 60)
print("PARKING VIOLATIONS — EDA")
print("=" * 60)
print(f"Total clean rows: {len(parking):,}")
print(f"Date range: {parking['created_datetime_ist'].min()} to {parking['created_datetime_ist'].max()}")
print(f"Unique vehicles: {parking['vehicle_number'].nunique():,}")
print(f"Unique police stations: {parking['police_station'].nunique()}")

print("\nTop 10 violation types:")
print(parking["violation_type"].value_counts().head(10))

print("\nVehicle type distribution:")
print(parking["vehicle_type"].value_counts().head(10))

print("\nViolations by hour of day (IST):")
hourly = parking["hour_ist"].value_counts().sort_index()
print(hourly)

print("\nViolations by day of week:")
print(parking["day_of_week"].value_counts())

# severity flag co-occurrence
print("\nSeverity flag totals:")
for col in ["is_main_road_parking", "is_no_parking", "is_wrong_parking", "is_footpath_parking"]:
    print(f"  {col}: {parking[col].sum():,} ({parking[col].mean()*100:.1f}%)")

# chart: violations by hour
plt.figure(figsize=(10, 4))
hourly.plot(kind="bar", color="#4C72B0")
plt.title("Parking Violations by Hour of Day (IST)")
plt.xlabel("Hour")
plt.ylabel("Violation count")
plt.tight_layout()
plt.savefig("eda_outputs/violations_by_hour.png", dpi=120)
plt.close()

print("\n" + "=" * 60)
print("ASTRAM INCIDENTS — EDA")
print("=" * 60)
print(f"Total clean rows: {len(incidents):,}")
print(f"Date range: {incidents['start_datetime_ist'].min()} to {incidents['start_datetime_ist'].max()}")

print("\nEvent cause distribution:")
print(incidents["event_cause"].value_counts())

print("\nStatus distribution:")
print(incidents["status"].value_counts())

print("\nroad_closure rate:")
print(incidents["requires_road_closure"].value_counts(normalize=True))

print("\nIncident duration stats (minutes, where computable):")
print(incidents["incident_duration_min"].describe())

print(f"\nRows with computable duration (ML training candidates): "
      f"{incidents['incident_duration_min'].notna().sum():,}")

# vehicle breakdown is dominant cause -- relevant for BreakdownBlind model framing
breakdown_pct = (incidents["event_cause"] == "vehicle_breakdown").mean() * 100
print(f"\nVehicle breakdown share of all incidents: {breakdown_pct:.1f}%")

# chart: incidents by hour
incidents["hour_ist"] = incidents["start_datetime_ist"].dt.hour
inc_hourly = incidents["hour_ist"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
inc_hourly.plot(kind="bar", color="#C44E52")
plt.title("Astram Incidents by Hour of Day (IST)")
plt.xlabel("Hour")
plt.ylabel("Incident count")
plt.tight_layout()
plt.savefig("eda_outputs/incidents_by_hour.png", dpi=120)
plt.close()

# chart: event cause breakdown
plt.figure(figsize=(8, 6))
incidents["event_cause"].value_counts().plot(kind="barh", color="#55A868")
plt.title("Incident Cause Breakdown")
plt.xlabel("Count")
plt.tight_layout()
plt.savefig("eda_outputs/incident_cause_breakdown.png", dpi=120)
plt.close()

print("\nSaved charts to eda_outputs/")
print(" - violations_by_hour.png")
print(" - incidents_by_hour.png")
print(" - incident_cause_breakdown.png")

# ---------------------------------------------------------------
# Geographic sanity check -- are lat/lng within Bangalore bounds?
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("GEOGRAPHIC SANITY CHECK")
print("=" * 60)
BLR_LAT = (12.7, 13.2)
BLR_LON = (77.4, 77.8)

p_in_bounds = parking["latitude"].between(*BLR_LAT) & parking["longitude"].between(*BLR_LON)
print(f"Parking rows within expected Bangalore bounds: {p_in_bounds.sum():,} / {len(parking):,} "
      f"({p_in_bounds.mean()*100:.1f}%)")

i_in_bounds = incidents["latitude"].between(*BLR_LAT) & incidents["longitude"].between(*BLR_LON)
print(f"Incident rows within expected Bangalore bounds: {i_in_bounds.sum():,} / {len(incidents):,} "
      f"({i_in_bounds.mean()*100:.1f}%)")

if p_in_bounds.mean() < 0.95 or i_in_bounds.mean() < 0.95:
    print("\n⚠️  WARNING: significant share of rows fall outside expected Bangalore lat/lng bounds.")
    print("   Investigate before running DBSCAN -- bad coordinates will distort clusters.")
else:
    print("\n✅ Coordinates look clean -- safe to proceed to DBSCAN clustering (Phase 2).")


PARKING VIOLATIONS — EDA
Total clean rows: 248,376
Date range: 2023-11-10 00:41:46+00:00 to 2024-04-08 23:00:46+00:00
Unique vehicles: 197,260
Unique police stations: 54

Top 10 violation types:
violation_type
["WRONG PARKING"]                                          113184
["NO PARKING"]                                             102353
["PARKING IN A MAIN ROAD","WRONG PARKING"]                   8016
["PARKING IN A MAIN ROAD","NO PARKING"]                      4080
["WRONG PARKING","DEFECTIVE NUMBER PLATE"]                   2549
["NO PARKING","PARKING IN A MAIN ROAD"]                      2087
["NO PARKING","DEFECTIVE NUMBER PLATE"]                      1939
["WRONG PARKING","PARKING IN A MAIN ROAD"]                   1618
["PARKING ON FOOTPATH","WRONG PARKING"]                      1006
["PARKING IN A MAIN ROAD","WRONG PARKING","NO PARKING"]       753
Name: count, dtype: int64

Vehicle type distribution:
vehicle_type
SCOOTER           78600
CAR               76758
MOTOR CYCLE    

## Step 5 — DBSCAN cluster the parking violations

**Input:  recidivism_scores.csv                                                        
Output: parking_clusters.geojson**

Runs DBSCAN on parking lat/lng using haversine distance, eps=100 metres,
min_samples=10. Every cluster is treated as a chronic illegal parking
hotspot. Computes per-cluster: violation count, dominant vehicle type,
dominant offence code, peak hour, recidivism multiplier average.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
IN_FILE = "recidivism_scores.csv"
OUT_FILE = "parking_clusters.geojson"
EARTH_RADIUS_KM = 6371.0
EPS_KM = 0.1  # 100 metres
MIN_SAMPLES = 10

print("Loading recidivism_scores.csv...")
df = pd.read_csv(IN_FILE)
print(f"Shape: {df.shape}")


Loading recidivism_scores.csv...
Shape: (248376, 34)


In [ ]:
# Run DBSCAN with haversine metric (requires coordinates in radians)

coords_rad = np.radians(df[["latitude", "longitude"]].values)
eps_rad = EPS_KM / EARTH_RADIUS_KM

print(f"\nRunning DBSCAN (eps={EPS_KM}km, min_samples={MIN_SAMPLES})...")
db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric="haversine")
df["cluster_id"] = db.fit_predict(coords_rad)

n_noise = (df["cluster_id"] == -1).sum()
n_clusters = df.loc[df["cluster_id"] != -1, "cluster_id"].nunique()
print(f"Clusters found: {n_clusters}")
print(f"Noise points (not in any cluster): {n_noise:,} ({n_noise/len(df)*100:.1f}%)")

clustered = df[df["cluster_id"] != -1].copy()
if len(clustered) == 0:
    raise RuntimeError("No clusters found -- eps/min_samples likely need tuning for this data's density.")



Running DBSCAN (eps=0.1km, min_samples=10)...
Clusters found: 529
Noise points (not in any cluster): 4,226 (1.7%)


In [ ]:
# Per-cluster aggregates

def mode_or_none(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else None

OFFENCE_COLS = ["is_main_road_parking", "is_no_parking", "is_wrong_parking", "is_footpath_parking"]


SEVERITY_WEIGHTS = {
    "is_main_road_parking": 5,
    "is_footpath_parking": 4,
    "is_no_parking": 3,
    "is_wrong_parking": 2,
}
clustered["violation_severity_score"] = sum(
    clustered[col] * weight for col, weight in SEVERITY_WEIGHTS.items()
)
print(f"\nPer-violation severity_score stats (accounts for stacked offences):")
print(clustered["violation_severity_score"].describe())

print("\nComputing per-cluster aggregates...")
cluster_summary = clustered.groupby("cluster_id").agg(
    violation_count=("id", "count"),
    centroid_lat=("latitude", "mean"),
    centroid_lon=("longitude", "mean"),
    dominant_vehicle_type=("vehicle_type", mode_or_none),
    peak_hour=("hour_ist", mode_or_none),
    avg_recidivism_tier=("recidivism_tier", "mean"),
    avg_violation_severity=("violation_severity_score", "mean"),
).reset_index()




Per-violation severity_score stats (accounts for stacked offences):
count    244150.000000
mean          2.960913
std           1.600825
min           2.000000
25%           2.000000
50%           3.000000
75%           3.000000
max          14.000000
Name: violation_severity_score, dtype: float64

Computing per-cluster aggregates...


In [ ]:
# dominant offence code = whichever severity flag has the highest sum in that cluster
offence_sums = clustered.groupby("cluster_id")[OFFENCE_COLS].sum()
cluster_summary["dominant_offence_code"] = cluster_summary["cluster_id"].map(offence_sums.idxmax(axis=1))

print(f"\nTotal compound clusters: {len(cluster_summary)}")
print(cluster_summary.describe(include="all"))


Total compound clusters: 529
        cluster_id  violation_count  centroid_lat  centroid_lon  \
count   529.000000       529.000000    529.000000    529.000000   
unique         NaN              NaN           NaN           NaN   
top            NaN              NaN           NaN           NaN   
freq           NaN              NaN           NaN           NaN   
mean    264.000000       461.531191     12.978283     77.603373   
std     152.853416      3371.004682      0.066521      0.064913   
min       0.000000         7.000000     12.806057     77.461264   
25%     132.000000        16.000000     12.931887     77.559577   
50%     264.000000        37.000000     12.973475     77.595434   
75%     396.000000       122.000000     13.021271     77.645089   
max     528.000000     64655.000000     13.248487     77.768123   

       dominant_vehicle_type   peak_hour  avg_recidivism_tier  \
count                    529  529.000000           529.000000   
unique                    16       

In [ ]:
# Save as GeoJSON

features = []
for _, row in cluster_summary.iterrows():
    props = row.drop(labels=["centroid_lat", "centroid_lon"]).to_dict()
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["centroid_lon"], row["centroid_lat"]]},
        "properties": props,
    })

geojson = {"type": "FeatureCollection", "features": features}
with open(OUT_FILE, "w") as f:
    json.dump(geojson, f, default=str)

print(f"\nSaved -> {OUT_FILE}")



Saved -> parking_clusters.geojson


## Step 6: DBSCAN cluster the Astram incidents
 **Input:  clean_incidents.csv                                                        
Output: incident_clusters.geojson**

Same DBSCAN approach as parking, applied to incident lat/lng. Per-cluster
computes: incident count, dominant event_cause, road_closure_rate (% that
required_road_closure=True), average incident duration, dominant vehicle
type involved.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

IN_FILE = "clean_incidents.csv"
OUT_FILE = "incident_clusters.geojson"

EARTH_RADIUS_KM = 6371.0
EPS_KM = 0.1  # 100 metres, same params as parking clustering
MIN_SAMPLES = 10

In [ ]:
# NOTE: tested min_samples=10 gives 60% noise (incidents dataset is much
# sparser than parking: 6.6K vs 248K rows). min_samples=5 cuts noise to ~33%
# (290 clusters) while still requiring a real cluster, not just 2-3 points.
# Kept at 10 here to match the parking dataset's parameters for consistency
# -- change to 5 below if you want denser incident cluster coverage for the
# spatial join in step 7.

print("Loading clean_incidents.csv...")
df = pd.read_csv(IN_FILE)
print(f"Shape: {df.shape}")


Loading clean_incidents.csv...
Shape: (6639, 53)


In [ ]:
# drop rows with missing coordinates defensively (none expected, but safe)
before = len(df)
df = df.dropna(subset=["latitude", "longitude"]).copy()
if len(df) != before:
    print(f"Dropped {before - len(df)} rows with missing coordinates")

coords_rad = np.radians(df[["latitude", "longitude"]].values)
eps_rad = EPS_KM / EARTH_RADIUS_KM

print(f"\nRunning DBSCAN (eps={EPS_KM}km, min_samples={MIN_SAMPLES})...")
db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric="haversine")
df["cluster_id"] = db.fit_predict(coords_rad)

n_noise = (df["cluster_id"] == -1).sum()
n_clusters = df.loc[df["cluster_id"] != -1, "cluster_id"].nunique()
print(f"Clusters found: {n_clusters}")
print(f"Noise points (not in any cluster): {n_noise:,} ({n_noise/len(df)*100:.1f}%)")

clustered = df[df["cluster_id"] != -1].copy()
if len(clustered) == 0:
    raise RuntimeError(
        "No clusters found with min_samples=10 -- the incidents dataset is "
        "much smaller than parking (6.6K vs 248K rows), so density may be "
        "too low for these params. Consider lowering min_samples (e.g. 5)."
    )


Running DBSCAN (eps=0.1km, min_samples=10)...
Clusters found: 114
Noise points (not in any cluster): 3,999 (60.2%)


In [ ]:
# Per-cluster aggregates
# ---------------------------------------------------------------
def mode_or_none(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else None

print("\nComputing per-cluster aggregates...")
cluster_summary = clustered.groupby("cluster_id").agg(
    incident_count=("id", "count"),
    centroid_lat=("latitude", "mean"),
    centroid_lon=("longitude", "mean"),
    dominant_event_cause=("event_cause", mode_or_none),
    road_closure_rate=("requires_road_closure", "mean"),
    avg_incident_duration=("incident_duration_min", "mean"),
    dominant_vehicle_type=("veh_type", mode_or_none),
).reset_index()

print(f"\nTotal incident clusters: {len(cluster_summary)}")
print(cluster_summary.head(10))



Computing per-cluster aggregates...

Total incident clusters: 114
   cluster_id  incident_count  centroid_lat  centroid_lon  \
0           0              64     13.040051     77.518387   
1           1             123     12.999458     77.584222   
2           2              16     12.993626     77.584701   
3           3              18     13.026333     77.544111   
4           4              16     12.936496     77.518155   
5           5             109     12.969367     77.700757   
6           6              32     12.966592     77.608285   
7           7              36     12.963909     77.584363   
8           8              17     12.947704     77.534068   
9           9              63     13.036461     77.589211   

  dominant_event_cause  road_closure_rate  avg_incident_duration  \
0    vehicle_breakdown           0.015625             107.180697   
1    vehicle_breakdown           0.040650             382.455026   
2    vehicle_breakdown           0.125000             940

In [ ]:
# Save as GeoJSON

features = []
for _, row in cluster_summary.iterrows():
    props = row.drop(labels=["centroid_lat", "centroid_lon"]).to_dict()
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["centroid_lon"], row["centroid_lat"]]},
        "properties": props,
    })

geojson = {"type": "FeatureCollection", "features": features}
with open(OUT_FILE, "w") as f:
    json.dump(geojson, f, default=str)

print(f"\nSaved -> {OUT_FILE}")



Saved -> incident_clusters.geojson


## Step 7: Spatial join both cluster sets
**Input:  parking_clusters.geojson, incident_clusters.geojson                      
Output: compound_zones.geojson**

For every parking cluster, finds all incident clusters within 200 metres.
Zones where both exist = compounding gridlock zones (joint evidence of
parking violations + traffic incidents at the same location).


In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

PARKING_FILE = "parking_clusters.geojson"
INCIDENT_FILE = "incident_clusters.geojson"
OUT_FILE = "compound_zones.geojson"

MAX_DISTANCE_M = 200
EARTH_RADIUS_M = 6371000

In [ ]:
def load_geojson_as_df(path):
    with open(path) as f:
        gj = json.load(f)
    rows = []
    for feat in gj["features"]:
        lon, lat = feat["geometry"]["coordinates"]
        row = {"lat": lat, "lon": lon, **feat["properties"]}
        rows.append(row)
    return pd.DataFrame(rows)

def latlon_to_unit_sphere(lat, lon):
    """Convert lat/lon (degrees) to xyz on a unit sphere for chord-distance KDTree search."""
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    x = np.cos(lat_r) * np.cos(lon_r)
    y = np.cos(lat_r) * np.sin(lon_r)
    z = np.sin(lat_r)
    return np.column_stack([x, y, z])


print("Loading cluster files...")
parking = load_geojson_as_df(PARKING_FILE)
incidents = load_geojson_as_df(INCIDENT_FILE)
print(f"Parking clusters: {len(parking)}")
print(f"Incident clusters: {len(incidents)}")

Loading cluster files...
Parking clusters: 529
Incident clusters: 114


In [ ]:
# build KDTree on incident cluster centroids (unit-sphere xyz)
incident_xyz = latlon_to_unit_sphere(incidents["lat"].values, incidents["lon"].values)
tree = cKDTree(incident_xyz)

parking_xyz = latlon_to_unit_sphere(parking["lat"].values, parking["lon"].values)


In [ ]:
# convert max distance (metres, great-circle) to equivalent unit-sphere chord distance
# chord = 2 * R * sin(theta/2), where theta = great_circle_dist / R
theta = MAX_DISTANCE_M / EARTH_RADIUS_M
max_chord = 2 * np.sin(theta / 2)

print(f"\nSearching for incident clusters within {MAX_DISTANCE_M}m of each parking cluster...")
dist_chord, idx = tree.query(parking_xyz, k=1, distance_upper_bound=max_chord)



Searching for incident clusters within 200m of each parking cluster...


In [ ]:
# convert chord distance back to metres for reporting
valid = np.isfinite(dist_chord)
dist_m = 2 * EARTH_RADIUS_M * np.arcsin(np.clip(dist_chord / 2, 0, 1))

print(f"Parking clusters with a matching incident cluster within {MAX_DISTANCE_M}m: "
      f"{valid.sum()} / {len(parking)}")

if valid.sum() == 0:
    raise RuntimeError(
        f"No compound zones found within {MAX_DISTANCE_M}m. Either the two "
        "datasets don't spatially overlap at this radius, or cluster eps "
        "needs tuning. Try increasing MAX_DISTANCE_M to sanity check."
    )


Parking clusters with a matching incident cluster within 200m: 45 / 529


In [ ]:
# Build the joined (compound zone) dataframe

compound = parking[valid].copy().reset_index(drop=True)
matched_incident_rows = incidents.iloc[idx[valid]].reset_index(drop=True)

# prefix columns to avoid name collisions, keep lat/lon from parking cluster
# (the "zone" location is anchored at the parking hotspot)
compound = compound.rename(columns={
    "cluster_id": "parking_cluster_id",
    "violation_count": "violation_count",
    "dominant_vehicle_type": "dominant_vehicle_type_parking",
    "dominant_offence_code": "dominant_offence_code",
    "peak_hour": "peak_hour",
    "avg_recidivism_tier": "avg_recidivism_tier",
})
matched_incident_rows = matched_incident_rows.rename(columns={
    "cluster_id": "incident_cluster_id",
    "incident_count": "incident_count",
    "dominant_event_cause": "dominant_event_cause",
    "road_closure_rate": "road_closure_rate",
    "avg_incident_duration": "avg_incident_duration",
    "dominant_vehicle_type": "dominant_vehicle_type_incident",
}).drop(columns=["lat", "lon"])

compound_zones = pd.concat([compound, matched_incident_rows], axis=1)
compound_zones["distance_to_incident_cluster_m"] = dist_m[valid]

print(f"\nTotal compound zones: {len(compound_zones)}")
print(compound_zones[[
    "parking_cluster_id", "incident_cluster_id", "violation_count",
    "incident_count", "distance_to_incident_cluster_m"
]].head(10))



Total compound zones: 45
   parking_cluster_id  incident_cluster_id  violation_count  incident_count  \
0                   0                   79              582              33   
1                   6                   70             3522               9   
2                  12                   24              198              47   
3                  37                   99              371              10   
4                  39                    9             3159              63   
5                  42                   12              319              51   
6                  53                   26              630              41   
7                  66                   72              900              23   
8                  68                   22             1454              59   
9                  71                   56              752              10   

   distance_to_incident_cluster_m  
0                       72.099448  
1                       43.24203

In [ ]:
# Save as GeoJSON
# ---------------------------------------------------------------
features = []
for _, row in compound_zones.iterrows():
    props = row.drop(labels=["lat", "lon"]).to_dict()
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
        "properties": props,
    })

geojson = {"type": "FeatureCollection", "features": features}
with open(OUT_FILE, "w") as f:
    json.dump(geojson, f, default=str)

print(f"\nSaved -> {OUT_FILE}")


Saved -> compound_zones.geojson


In [ ]:
# ---------------------------------------------------------------
# ALTERNATIVE: if geopandas IS available on your machine, this is the
# more direct equivalent (commented out, not run here):
#
# import geopandas as gpd
# parking_gdf = gpd.read_file(PARKING_FILE).to_crs(epsg=32643)
# incidents_gdf = gpd.read_file(INCIDENT_FILE).to_crs(epsg=32643)
# joined = gpd.sjoin_nearest(parking_gdf, incidents_gdf, max_distance=200,
#                             distance_col="dist_m", how="inner")
# joined.to_crs(epsg=4326).to_file(OUT_FILE, driver="GeoJSON")
# ---------------------------------------------------------------

## Step 8: Compute Compound Risk Score (CRS) per zone
**Input:  compound_zones.geojson                                   
Output: crs_scores.csv**

CRS = (vehicle_weight x violation_severity x recidivism_factor)
      x (1 + incident_count/10 x road_closure_flag)

Vehicle weights (slide spec): HGV=5, Bus=4, Car=2, Auto=1.5, Scooter=1.
Violation severity (slide spec): main_road=5, footpath=4, no_parking=3,
wrong_parking=2.

In [ ]:
import json
import pandas as pd

IN_FILE = "compound_zones.geojson"
OUT_FILE = "crs_scores.csv"


In [ ]:
# Vehicle weight mapping (slide spec: HGV=5, Bus=4, Car=2, Auto=1.5, Scooter=1)
# Extended to cover every real category found in the data.

VEHICLE_WEIGHTS_PARKING = {
    "HGV": 5, "LORRY/GOODS VEHICLE": 5, "TANKER": 5, "TRACTOR": 5, "MINI LORRY": 5,
    "BUS (BMTC/KSRTC)": 4, "PRIVATE BUS": 4, "TOURIST BUS": 4, "FACTORY BUS": 4, "SCHOOL VEHICLE": 4,
    "CAR": 2, "JEEP": 2, "VAN": 2, "LGV": 2, "TEMPO": 2,
    "PASSENGER AUTO": 1.5, "GOODS AUTO": 1.5, "MAXI-CAB": 1.5,
    "SCOOTER": 1, "MOTOR CYCLE": 1, "MOPED": 1, "OTHERS": 1,
}

VEHICLE_WEIGHTS_INCIDENT = {
    "heavy_vehicle": 5, "truck": 5,
    "bmtc_bus": 4, "ksrtc_bus": 4, "private_bus": 4,
    "private_car": 2, "taxi": 2, "lcv": 2,
    "auto": 1.5,
    "others": 1,
}


In [ ]:
print("Loading compound_zones.geojson...")
with open(IN_FILE) as f:
    gj = json.load(f)

rows = []
for feat in gj["features"]:
    lon, lat = feat["geometry"]["coordinates"]
    rows.append({"lat": lat, "lon": lon, **feat["properties"]})
df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")

Loading compound_zones.geojson...
Shape: (45, 16)


In [ ]:
# vehicle_weight: use the parking cluster's dominant vehicle type
# (the zone is anchored at the parking hotspot, so this is the more
# relevant "who's causing the chronic violations here" signal)
# ---------------------------------------------------------------
df["vehicle_weight"] = df["dominant_vehicle_type_parking"].map(VEHICLE_WEIGHTS_PARKING)
unmapped = df["vehicle_weight"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows had unmapped vehicle types, defaulting to weight=1")
    print("Unmapped values:", df.loc[df["vehicle_weight"].isna(), "dominant_vehicle_type_parking"].unique())
df["vehicle_weight"] = df["vehicle_weight"].fillna(1)

In [ ]:
df["violation_severity"] = df["avg_violation_severity"]

# ---------------------------------------------------------------
# recidivism_factor: average recidivism tier of vehicles in this zone
# ---------------------------------------------------------------
df["recidivism_factor"] = df["avg_recidivism_tier"]


# CRS formula
# road_closure_flag = road_closure_rate (0-1), already computed per
# incident cluster in step 6 as the fraction of incidents requiring closure
# ---------------------------------------------------------------
df["road_closure_flag"] = df["road_closure_rate"]

df["raw_crs"] = (
    df["vehicle_weight"] * df["violation_severity"] * df["recidivism_factor"]
    * (1 + df["incident_count"] / 10 * df["road_closure_flag"])
)


In [ ]:
# Normalise to 0-100
# ---------------------------------------------------------------
crs_min, crs_max = df["raw_crs"].min(), df["raw_crs"].max()
if crs_max == crs_min:
    print("WARNING: all raw_crs values identical -- normalised CRS set to 50 for all rows")
    df["CRS"] = 50.0
else:
    df["CRS"] = (df["raw_crs"] - crs_min) / (crs_max - crs_min) * 100

print(f"\nCRS distribution:")
print(df["CRS"].describe())

print(f"\nTop 5 highest-risk zones:")
top5 = df.nlargest(5, "CRS")[[
    "parking_cluster_id", "incident_cluster_id", "CRS",
    "violation_count", "incident_count", "dominant_vehicle_type_parking",
    "dominant_offence_code", "road_closure_rate"
]]
print(top5.to_string(index=False))


CRS distribution:
count     45.000000
mean      29.920941
std       24.535795
min        0.000000
25%        8.647860
50%       27.956111
75%       44.641225
max      100.000000
Name: CRS, dtype: float64

Top 5 highest-risk zones:
 parking_cluster_id  incident_cluster_id        CRS  violation_count  incident_count dominant_vehicle_type_parking dominant_offence_code  road_closure_rate
                283                  107 100.000000               35              13                           CAR      is_wrong_parking           0.461538
                247                    5  75.936310              140             109                           CAR         is_no_parking           0.018349
                206                   95  70.521532              194              10                           CAR      is_wrong_parking           0.000000
                463                   65  64.306429               15              36                           CAR      is_wrong_parking        

In [ ]:
# Save

df.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")
print(f"Total zones with CRS scores: {len(df)}")


Saved -> crs_scores.csv
Total zones with CRS scores: 45


## Step 9: Hardcode Bangalore hospital coordinates
Output: hospitals.json

8 major hospitals, no API needed -- coordinates as specified in the plan.
"""

In [ ]:
import json

HOSPITALS = [
    {"name": "Manipal Hospital", "lat": 13.0048, "lon": 77.5564},
    {"name": "Fortis Hospital", "lat": 12.9291, "lon": 77.6869},
    {"name": "Bowring & Lady Curzon Hospital", "lat": 12.9716, "lon": 77.6046},
    {"name": "Victoria Hospital", "lat": 12.9659, "lon": 77.5752},
    {"name": "St. John's Medical College Hospital", "lat": 12.9249, "lon": 77.6187},
    {"name": "MS Ramaiah Memorial Hospital", "lat": 13.0195, "lon": 77.5547},
    {"name": "Sakra World Hospital", "lat": 12.9358, "lon": 77.6762},
    {"name": "Apollo Hospital", "lat": 12.8456, "lon": 77.6603},
]

with open("hospitals.json", "w") as f:
    json.dump(HOSPITALS, f, indent=2)

print(f"Saved {len(HOSPITALS)} hospitals -> hospitals.json")
for h in HOSPITALS:
    print(f"  {h['name']}: ({h['lat']}, {h['lon']})")

Saved 8 hospitals -> hospitals.json
  Manipal Hospital: (13.0048, 77.5564)
  Fortis Hospital: (12.9291, 77.6869)
  Bowring & Lady Curzon Hospital: (12.9716, 77.6046)
  Victoria Hospital: (12.9659, 77.5752)
  St. John's Medical College Hospital: (12.9249, 77.6187)
  MS Ramaiah Memorial Hospital: (13.0195, 77.5547)
  Sakra World Hospital: (12.9358, 77.6762)
  Apollo Hospital: (12.8456, 77.6603)


## Step 10: Compute ambulance proximity weight per zone

 **Input:  crs_scores.csv, hospitals.json                                        
Output: final_priority_scores.csv**                       
For each compound zone, computes haversine distance to every hospital.
ambulance_weight tiers:                           closest hospital
  
                    

*   within 1km  ->                  3.0  


*    within 2km                   -> 2.0
*   beyond 2km                   -> 1.0




final_priority_score = CRS * ambulance_weight (overrides raw CRS ranking --
zones near hospitals jump to the top regardless of raw CRS).

In [ ]:
import json
import numpy as np
import pandas as pd

CRS_FILE = "crs_scores.csv"
HOSPITALS_FILE = "hospitals.json"
OUT_FILE = "final_priority_scores.csv"

EARTH_RADIUS_KM = 6371.0

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km between one point and an array of points."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c


print("Loading crs_scores.csv and hospitals.json...")
df = pd.read_csv(CRS_FILE)
with open(HOSPITALS_FILE) as f:
    hospitals = json.load(f)

hosp_lats = np.array([h["lat"] for h in hospitals])
hosp_lons = np.array([h["lon"] for h in hospitals])
hosp_names = [h["name"] for h in hospitals]

print(f"Zones: {len(df)}, Hospitals: {len(hospitals)}")

Loading crs_scores.csv and hospitals.json...
Zones: 45, Hospitals: 8


In [ ]:
# For each zone, find distance to nearest hospital

nearest_dist_km = []
nearest_hosp_name = []

for _, row in df.iterrows():
    dists = haversine_km(row["lat"], row["lon"], hosp_lats, hosp_lons)
    min_idx = np.argmin(dists)
    nearest_dist_km.append(dists[min_idx])
    nearest_hosp_name.append(hosp_names[min_idx])

df["nearest_hospital"] = nearest_hosp_name
df["nearest_hospital_dist_km"] = nearest_dist_km


def ambulance_weight(dist_km):
    if dist_km <= 1.0:
        return 3.0
    elif dist_km <= 2.0:
        return 2.0
    else:
        return 1.0


df["ambulance_weight"] = df["nearest_hospital_dist_km"].apply(ambulance_weight)

print("\nambulance_weight distribution:")
print(df["ambulance_weight"].value_counts().sort_index())



ambulance_weight distribution:
ambulance_weight
1.0    40
2.0     3
3.0     2
Name: count, dtype: int64


In [ ]:
# final_priority_score = CRS * ambulance_weight (overrides raw CRS ranking)

df["final_priority_score"] = df["CRS"] * df["ambulance_weight"]

print("\nfinal_priority_score stats:")
print(df["final_priority_score"].describe())

print("\nTop 10 zones by final_priority_score (note how ambulance proximity reorders vs raw CRS):")
top10 = df.nlargest(10, "final_priority_score")[[
    "parking_cluster_id", "CRS", "ambulance_weight", "final_priority_score",
    "nearest_hospital", "nearest_hospital_dist_km"
]]
print(top10.to_string(index=False))



final_priority_score stats:
count     45.000000
mean      35.340418
std       32.457088
min        0.000000
25%        8.647860
50%       28.038079
75%       57.932024
max      141.043065
Name: final_priority_score, dtype: float64

Top 10 zones by final_priority_score (note how ambulance proximity reorders vs raw CRS):
 parking_cluster_id        CRS  ambulance_weight  final_priority_score                    nearest_hospital  nearest_hospital_dist_km
                206  70.521532               2.0            141.043065 St. John's Medical College Hospital                  1.752520
                  0  36.097541               3.0            108.292624 St. John's Medical College Hospital                  0.083485
                283 100.000000               1.0            100.000000        MS Ramaiah Memorial Hospital                  6.843767
                328  44.641225               2.0             89.282451      Bowring & Lady Curzon Hospital                  1.577672
             

In [ ]:
# sanity check: did ranking actually change vs raw CRS?
crs_rank = df["CRS"].rank(ascending=False)
priority_rank = df["final_priority_score"].rank(ascending=False)
n_reordered = (crs_rank != priority_rank).sum()
print(f"\nZones whose rank changed due to ambulance weighting: {n_reordered} / {len(df)}")

df.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")


Zones whose rank changed due to ambulance weighting: 32 / 45

Saved -> final_priority_scores.csv


## Step 11: Flag active road-closure incidents near hospitals
 **Input:  clean_incidents.csv, hospitals.json                     
Output: ambulance_alerts.json**

Filters incidents where status='active' AND requires_road_closure=True.
For each, checks hospital proximity. These become CRITICAL AMBULANCE
ALERT entries -- shown in red on the dashboard above all other priorities.
This is the live, real-time feed used during the demo replay.

In [ ]:
import json
import numpy as np
import pandas as pd

INCIDENTS_FILE = "clean_incidents.csv"
HOSPITALS_FILE = "hospitals.json"
OUT_FILE = "ambulance_alerts.json"

EARTH_RADIUS_KM = 6371.0


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c


print("Loading incidents and hospitals...")
df = pd.read_csv(INCIDENTS_FILE)
with open(HOSPITALS_FILE) as f:
    hospitals = json.load(f)

hosp_lats = np.array([h["lat"] for h in hospitals])
hosp_lons = np.array([h["lon"] for h in hospitals])
hosp_names = [h["name"] for h in hospitals]

print(f"Total incidents: {len(df)}")
print(f"status distribution:\n{df['status'].value_counts(dropna=False)}")
print(f"\nrequires_road_closure distribution:\n{df['requires_road_closure'].value_counts(dropna=False)}")

Loading incidents and hospitals...
Total incidents: 6639
status distribution:
status
closed      5581
active       993
resolved      65
Name: count, dtype: int64

requires_road_closure distribution:
requires_road_closure
False    6105
True      534
Name: count, dtype: int64


In [ ]:
# Filter: status == 'active' AND requires_road_closure == True
# ---------------------------------------------------------------
critical = df[(df["status"] == "active") & (df["requires_road_closure"] == True)].copy()
print(f"\nCRITICAL AMBULANCE ALERT candidates: {len(critical)}")

if len(critical) == 0:
    print("WARNING: no rows matched the active+road_closure filter. "
          "Check that 'status' and 'requires_road_closure' values/dtypes "
          "match what's expected -- see distributions printed above.")



CRITICAL AMBULANCE ALERT candidates: 100


In [ ]:
# Check hospital proximity for each
# ---------------------------------------------------------------
alerts = []
for _, row in critical.iterrows():
    dists = haversine_km(row["latitude"], row["longitude"], hosp_lats, hosp_lons)
    min_idx = np.argmin(dists)
    alerts.append({
        "incident_id": row["id"],
        "lat": row["latitude"],
        "lon": row["longitude"],
        "event_cause": row["event_cause"],
        "veh_type": row.get("veh_type", None),
        "start_datetime_ist": str(row.get("start_datetime_ist", "")),
        "nearest_hospital": hosp_names[min_idx],
        "nearest_hospital_dist_km": round(float(dists[min_idx]), 3),
        "alert_level": "CRITICAL_AMBULANCE_ALERT",
    })

In [ ]:
# sort by proximity to hospital -- most urgent (closest) first
alerts.sort(key=lambda a: a["nearest_hospital_dist_km"])

print(f"\nTop 5 most urgent alerts (closest to a hospital):")
for a in alerts[:5]:
    print(f"  {a['incident_id']}: {a['event_cause']} - {a['nearest_hospital_dist_km']}km from {a['nearest_hospital']}")

with open(OUT_FILE, "w") as f:
    json.dump(alerts, f, indent=2, default=str)

print(f"\nSaved {len(alerts)} alerts -> {OUT_FILE}")



Top 5 most urgent alerts (closest to a hospital):
  FKID003234: water_logging - 0.2km from Victoria Hospital
  FKID002664: road_conditions - 0.616km from Victoria Hospital
  FKID001517: others - 0.644km from Bowring & Lady Curzon Hospital
  FKID003477: construction - 0.779km from Bowring & Lady Curzon Hospital
  FKID003707: construction - 0.779km from Bowring & Lady Curzon Hospital

Saved 100 alerts -> ambulance_alerts.json


## Step 12: Build training dataset for duration model
Input:  clean_incidents.csv                                    
Output: duration_train.csv

Uses rows where incident_duration is computable (closed_datetime exists).
In the real data this is 1,885 rows (28.4% of clean incidents) -- lower
than the plan's ~3,000 estimate, but that's the true count after the
72hr/negative-duration cleanup from step 2. Still a workable training set.

Features (7, per plan): event_cause, veh_type, corridor, requires_road_closure,
priority, hour_of_day (IST), day_of_week.
Target: incident_duration in minutes.

In [ ]:
import pandas as pd

IN_FILE = "clean_incidents.csv"
OUT_FILE = "duration_train.csv"

FEATURES = ["event_cause", "veh_type", "corridor", "requires_road_closure",
            "priority", "hour_ist", "day_of_week"]
TARGET = "incident_duration_min"

print("Loading clean_incidents.csv...")
df = pd.read_csv(IN_FILE)
print(f"Total rows: {len(df)}")

# rows where duration is computable
train_df = df[df[TARGET].notna()].copy()
print(f"Rows with computable duration: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")

Loading clean_incidents.csv...
Total rows: 6639
Rows with computable duration: 1885 (28.4%)


In [ ]:
# veh_type has real NaNs (~34% in this subset) -- fill with explicit
# 'unknown' category rather than dropping rows, since dropping would
# shrink an already-small training set further
missing_veh = train_df["veh_type"].isna().sum()
print(f"veh_type missing: {missing_veh} ({missing_veh/len(train_df)*100:.1f}%) -- filling with 'unknown'")
train_df["veh_type"] = train_df["veh_type"].fillna("unknown")

veh_type missing: 646 (34.3%) -- filling with 'unknown'


In [ ]:
# corridor / priority -- check for any missing too, fill defensively
for col in ["corridor", "priority", "event_cause"]:
    n_missing = train_df[col].isna().sum()
    if n_missing > 0:
        print(f"{col} missing: {n_missing} -- filling with 'unknown'")
        train_df[col] = train_df[col].fillna("unknown")

train_final = train_df[FEATURES + [TARGET]].copy()

print(f"\nFinal training set shape: {train_final.shape}")
print(f"\nTarget ({TARGET}) distribution:")
print(train_final[TARGET].describe())

print(f"\nFeature summary:")
for col in FEATURES:
    n_unique = train_final[col].nunique()
    print(f"  {col}: {n_unique} unique values, dtype={train_final[col].dtype}")

train_final.to_csv(OUT_FILE, index=False)
print(f"\nSaved -> {OUT_FILE}")

corridor missing: 1 -- filling with 'unknown'

Final training set shape: (1885, 8)

Target (incident_duration_min) distribution:
count    1885.000000
mean      284.205640
std       687.671327
min         1.678647
25%        30.688685
50%        61.567407
75%       112.377128
max      4312.246381
Name: incident_duration_min, dtype: float64

Feature summary:
  event_cause: 10 unique values, dtype=object
  veh_type: 11 unique values, dtype=object
  corridor: 23 unique values, dtype=object
  requires_road_closure: 2 unique values, dtype=bool
  priority: 2 unique values, dtype=object
  hour_ist: 22 unique values, dtype=int64
  day_of_week: 7 unique values, dtype=object

Saved -> duration_train.csv


## Step 13: Train duration regression model
Input:  duration_train.csv             
Output: duration_model.pkl


In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMClassifier
IN_FILE = "duration_train.csv"
MODEL_FILE = "duration_model.pkl"

FEATURES = ["event_cause", "veh_type", "corridor", "requires_road_closure",
            "priority", "hour_ist", "day_of_week"]
CATEGORICAL_COLS = ["event_cause", "veh_type", "corridor", "priority", "day_of_week"]
TARGET = "incident_duration_min"

print("Loading duration_train.csv...")
df = pd.read_csv(IN_FILE)
print(f"Shape: {df.shape}")

Loading duration_train.csv...
Shape: (1885, 8)


In [ ]:
# Label encode categoricals
# ---------------------------------------------------------------
encoders = {}
df_enc = df.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le

# requires_road_closure is already boolean -- convert to int
df_enc["requires_road_closure"] = df_enc["requires_road_closure"].astype(int)

X = df_enc[FEATURES]
y_raw = df_enc[TARGET]

In [ ]:
y = np.log1p(y_raw)

# ---------------------------------------------------------------
# 80/20 train-val split
# ---------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_val_raw = np.expm1(y_val)  # back to actual minutes, for honest MAE reporting
print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}")



Train: 1508, Val: 377


In [ ]:
# Train model
import lightgbm as lgb
model = lgb.LGBMRegressor(random_state=42, verbosity=-1)
print("\nTraining LightGBM...")
model.fit(X_train, y_train)


Training LightGBM...


LGBMRegressor(random_state=42, verbosity=-1)

In [ ]:
# Evaluate (back-transform predictions from log-space to minutes)
# ---------------------------------------------------------------
preds_train_raw = np.expm1(model.predict(X_train))
preds_val_raw = np.expm1(model.predict(X_val))
y_train_raw = np.expm1(y_train)

mae_train = mean_absolute_error(y_train_raw, preds_train_raw)
mae_val = mean_absolute_error(y_val_raw, preds_val_raw)

print(f"\nTrain MAE: {mae_train:.2f} minutes")
print(f"Val MAE:   {mae_val:.2f} minutes")
print(f"(For reference, mean duration is {y_raw.mean():.1f} min, "
      f"median is {y_raw.median():.1f} min -- compare MAE against these)")

print("\n*** HONEST LIMITATION ***")
print(f"90th percentile of actual duration is {y_raw.quantile(0.9):.0f} min, but the")
print(f"median is only {y_raw.median():.0f} min -- the data has a heavy long tail")
print(f"({(y_raw>500).sum()} of {len(y_raw)} incidents run over 500 min). With only")
print(f"~1,885 training rows, the model can't reliably separate 'routine 60-min")
print(f"breakdown' from 'multi-hour incident' on these 7 features alone. Treat")
print(f"predictions as a rough relative signal (which zones are likely SLOWER")
print(f"than others) rather than a precise minute-level estimate in the demo.")


Train MAE: 185.06 minutes
Val MAE:   286.10 minutes
(For reference, mean duration is 284.2 min, median is 61.6 min -- compare MAE against these)

*** HONEST LIMITATION ***
90th percentile of actual duration is 871 min, but the
median is only 62 min -- the data has a heavy long tail
(222 of 1885 incidents run over 500 min). With only
~1,885 training rows, the model can't reliably separate 'routine 60-min
breakdown' from 'multi-hour incident' on these 7 features alone. Treat
predictions as a rough relative signal (which zones are likely SLOWER
than others) rather than a precise minute-level estimate in the demo.


In [ ]:
# Check expected signal (per plan): BMTC bus breakdowns take longer than
# private cars; road_closure=True takes longer; Mysore Road longer than
# inner streets
# ---------------------------------------------------------------
print("\nSanity check against expected real-world signal:")
bmtc_mask = df["veh_type"] == "bmtc_bus"
car_mask = df["veh_type"] == "private_car"
if bmtc_mask.sum() > 0 and car_mask.sum() > 0:
    print(f"  Avg duration, bmtc_bus breakdowns: {df.loc[bmtc_mask, TARGET].mean():.1f} min")
    print(f"  Avg duration, private_car breakdowns: {df.loc[car_mask, TARGET].mean():.1f} min")

closure_mask = df["requires_road_closure"] == True
no_closure_mask = df["requires_road_closure"] == False
print(f"  Avg duration, road_closure=True: {df.loc[closure_mask, TARGET].mean():.1f} min")
print(f"  Avg duration, road_closure=False: {df.loc[no_closure_mask, TARGET].mean():.1f} min")

mysore_mask = df["corridor"] == "Mysore Road"
if mysore_mask.sum() > 0:
    print(f"  Avg duration, Mysore Road: {df.loc[mysore_mask, TARGET].mean():.1f} min")
    print(f"  Avg duration, overall: {df[TARGET].mean():.1f} min")


Sanity check against expected real-world signal:
  Avg duration, bmtc_bus breakdowns: 61.2 min
  Avg duration, private_car breakdowns: 41.8 min
  Avg duration, road_closure=True: 409.1 min
  Avg duration, road_closure=False: 273.7 min
  Avg duration, Mysore Road: 174.6 min
  Avg duration, overall: 284.2 min


In [ ]:
# Feature importance

print("\nFeature importances:")
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importances)



Feature importances:
hour_ist                 797
corridor                 598
day_of_week              594
veh_type                 439
event_cause              435
priority                  74
requires_road_closure     63
dtype: int32


In [ ]:
# Save model + encoders together (encoders needed at inference time)

with open(MODEL_FILE, "wb") as f:
    pickle.dump({"model": model, "encoders": encoders, "features": FEATURES,
                 "categorical_cols": CATEGORICAL_COLS,
                 "target_is_log_transformed": True}, f)

print(f"\nSaved -> {MODEL_FILE}")


Saved -> duration_model.pkl


## Step 14: Attach duration predictions to compound zones
Input:  final_priority_scores.csv, duration_model.pkl, clean_incidents.csv       
Output: live_zone_status.json

For every compound zone, runs duration_model.pkl to predict how many more
minutes the matched incident cluster will likely block traffic. This feeds
the ambulance delay estimate on the dashboard, e.g.:
  "Mekhri Circle: blocked est. 34 more min. Ambulance delay: CRITICAL."

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

ZONES_FILE = "final_priority_scores.csv"
MODEL_FILE = "duration_model.pkl"
INCIDENTS_FILE = "clean_incidents.csv"
OUT_FILE = "live_zone_status.json"

EPS_KM = 0.1
MIN_SAMPLES = 10
EARTH_RADIUS_KM = 6371.0

In [ ]:
def mode_or_none(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else None


print("Loading compound zones...")
zones = pd.read_csv(ZONES_FILE)
print(f"Zones: {len(zones)}")

print("\nRe-deriving corridor/priority/hour/day per incident cluster "
      "(same DBSCAN params as step 6, to recover features step 6 didn't "
      "carry forward)...")
incidents = pd.read_csv(INCIDENTS_FILE)
coords_rad = np.radians(incidents[["latitude", "longitude"]].values)
eps_rad = EPS_KM / EARTH_RADIUS_KM
db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric="haversine")
incidents["cluster_id"] = db.fit_predict(coords_rad)

extra_features = (
    incidents[incidents["cluster_id"] != -1]
    .groupby("cluster_id")
    .agg(
        corridor=("corridor", mode_or_none),
        priority=("priority", mode_or_none),
        hour_ist=("hour_ist", mode_or_none),
        day_of_week=("day_of_week", mode_or_none),
    )
    .reset_index()
    .rename(columns={"cluster_id": "incident_cluster_id"})
)
print(f"Recovered features for {len(extra_features)} incident clusters")

zones = zones.merge(extra_features, on="incident_cluster_id", how="left")
missing = zones[["corridor", "priority", "hour_ist", "day_of_week"]].isna().any(axis=1).sum()
if missing > 0:
    print(f"WARNING: {missing} zones missing re-derived features after merge -- "
          f"filling with safe defaults (most common training value)")

Loading compound zones...
Zones: 45

Re-deriving corridor/priority/hour/day per incident cluster (same DBSCAN params as step 6, to recover features step 6 didn't carry forward)...
Recovered features for 114 incident clusters


In [ ]:
# Load model

print("\nLoading duration_model.pkl...")
with open(MODEL_FILE, "rb") as f:
    bundle = pickle.load(f)
model = bundle["model"]
encoders = bundle["encoders"]
features = bundle["features"]
categorical_cols = bundle["categorical_cols"]
is_log = bundle.get("target_is_log_transformed", False)



Loading duration_model.pkl...


In [ ]:
# Build prediction feature frame, mapping compound-zone columns to the model's expected feature names

pred_df = pd.DataFrame({
    "event_cause": zones["dominant_event_cause"],
    "veh_type": zones["dominant_vehicle_type_incident"],
    "corridor": zones["corridor"],
    "requires_road_closure": (zones["road_closure_rate"] > 0.5),  # majority-closure proxy
    "priority": zones["priority"],
    "hour_ist": zones["hour_ist"],
    "day_of_week": zones["day_of_week"],
})


In [ ]:
# fill any remaining NaNs with safe fallback values seen during training
for col in categorical_cols:
    pred_df[col] = pred_df[col].fillna("unknown")
pred_df["hour_ist"] = pred_df["hour_ist"].fillna(pred_df["hour_ist"].median() if pred_df["hour_ist"].notna().any() else 12)
pred_df["requires_road_closure"] = pred_df["requires_road_closure"].astype(int)

In [ ]:
# apply the SAME label encoders fit during training -- unseen categories at inference time get mapped to a safe fallback (0) rather than crashing
for col in categorical_cols:
    le = encoders[col]
    known = set(le.classes_)
    pred_df[col] = pred_df[col].apply(lambda v: v if v in known else le.classes_[0])
    pred_df[col] = le.transform(pred_df[col].astype(str))

X_pred = pred_df[features]

In [ ]:
# Predict (back-transform from log-space since model was trained on log1p)

raw_preds = model.predict(X_pred)
zones["predicted_remaining_block_min"] = np.expm1(raw_preds) if is_log else raw_preds
zones["predicted_remaining_block_min"] = zones["predicted_remaining_block_min"].round(1)

print("\nPredicted blockage duration distribution:")
print(zones["predicted_remaining_block_min"].describe())


Predicted blockage duration distribution:
count     45.000000
mean      54.284444
std       20.690786
min       26.500000
25%       41.100000
50%       49.500000
75%       60.300000
max      137.400000
Name: predicted_remaining_block_min, dtype: float64


In [ ]:
# Ambulance delay severity label, combining duration prediction +
# ambulance_weight from Phase 3 (step 10)

def delay_label(row):
    if row["ambulance_weight"] >= 3.0 and row["predicted_remaining_block_min"] >= 20:
        return "CRITICAL"
    elif row["ambulance_weight"] >= 2.0 and row["predicted_remaining_block_min"] >= 15:
        return "HIGH"
    elif row["predicted_remaining_block_min"] >= 10:
        return "MODERATE"
    else:
        return "LOW"

zones["ambulance_delay_label"] = zones.apply(delay_label, axis=1)

print("\nambulance_delay_label distribution:")
print(zones["ambulance_delay_label"].value_counts())

print("\nExample dashboard lines:")
for _, row in zones.nlargest(3, "final_priority_score").iterrows():
    print(f"  Zone {row['parking_cluster_id']}: blocked est. "
          f"{row['predicted_remaining_block_min']:.0f} more min. "
          f"Ambulance delay: {row['ambulance_delay_label']}.")


ambulance_delay_label distribution:
ambulance_delay_label
MODERATE    40
HIGH         3
CRITICAL     2
Name: count, dtype: int64

Example dashboard lines:
  Zone 206: blocked est. 40 more min. Ambulance delay: HIGH.
  Zone 0: blocked est. 137 more min. Ambulance delay: CRITICAL.
  Zone 283: blocked est. 36 more min. Ambulance delay: MODERATE.


In [ ]:
# Save as live_zone_status.json

zones_out = zones.drop(columns=["corridor", "priority", "hour_ist", "day_of_week"])
zones_out.to_json(OUT_FILE, orient="records", indent=2)

print(f"\nSaved -> {OUT_FILE}")
print(f"Total zones with live status: {len(zones)}")


Saved -> live_zone_status.json
Total zones with live status: 45
